# Signals Example

This notebook previews several standalone acoustic source and noise models, then combines them into a simple composite soundscape. It is intended as a quick orientation notebook for how different source classes behave before they are embedded in a full propagation and beamforming pipeline.

**Background**
- Passive-sonar scenes often contain a mix of biological, anthropogenic, and ambient contributors rather than a single clean source.
- Understanding the isolated time-frequency signature of each component makes it easier to interpret later BTRs, spectrograms, and received mixtures.
- A lightweight soundscape example is also useful for validating source classes without introducing propagation or tracking complexity.

**Key Concepts**
- Standalone signal generation for biological, anthropogenic, and ambient sources.
- Spectrogram-based inspection of time-frequency structure.
- Building a composite soundscape from individually interpretable components.


## Setup and Reproducibility

This section imports the shared dependencies, fixes the random seed, and defines one helper used to preview each generated signal with a spectrogram and audio widget.


In [ ]:
from datetime import datetime

import numpy as np
from IPython.display import Audio, display
from stonesoup.types.groundtruth import GroundTruthState

from bluepebble.plotter import plot_spectrogram
from bluepebble.signal.anthropogenic import BroadbandRecordedSignal, NarrowbandTonalSignal
from bluepebble.signal.biological import PointSourceSnappingShrimpSignal, WhaleCallSignal
from bluepebble.signal.effects import Reverb
from bluepebble.signal.random import WhiteNoise

seed = 2000
np.random.seed(seed)

SAMPLING_RATE_HZ = 48_000
SIGNAL_DURATION_S = 10.0
REFERENCE_TIME = datetime(2026, 1, 1, 0, 0, 0)

component_signals = {}


def preview_signal(
    signal: np.ndarray,
    title: str,
    n_fft: int,
    hop_length: int,
    y_lim: tuple[float, float],
    yaxis_format: str = "kHz",
) -> None:
    """Show a spectrogram and inline audio preview for a generated signal."""
    fig = plot_spectrogram(
        signal,
        SAMPLING_RATE_HZ,
        n_fft=n_fft,
        hop_length=hop_length,
        y_lim=y_lim,
        yaxis_format=yaxis_format,
    )
    fig.update_layout(title=title)
    fig.show()
    display(Audio(data=np.asarray(signal), rate=SAMPLING_RATE_HZ))

## Whale Call Signal

This section simulates a structured humpback-style vocalisation.

- The call is built from reusable themes and phrases.
- Harmonics, vibrato, breathy noise, and reverb shape the timbre.
- The result is a comparatively rich mid-frequency biological source.


In [ ]:
whale_source = GroundTruthState(
    [0, 0, 0, 0],
    timestamp=REFERENCE_TIME,
    metadata={
        "amplitude_upa": 10 ** (180 / 20),
    },
)

theme_0 = [15, -15, 15, -15]
theme_1 = [10, 0]
theme_2 = [-20, -10, 10]
theme_3 = [0, 0, 0, 0]

phrase_a = [0, 1]
phrase_b = [2, 1]
phrase_c = [3]
song_phrases = [phrase_c, phrase_a, phrase_b]

reverb_effect = Reverb(duration_s=0.4, wet_dry_mix=0.8)

whale_signal_model = WhaleCallSignal(
    duration_s=SIGNAL_DURATION_S,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    song_structure_enabled=True,
    song_themes=[theme_0, theme_1, theme_2, theme_3],
    song_phrases=song_phrases,
    theme_base_freq_hz=250,
    theme_freq_jitter_hz=10,
    theme_duration_s=1.2,
    mean_call_interval_s=2.0,
    interval_jitter_s=0.3,
    call_duration_s=1.0,
    duration_jitter_s=0.3,
    start_freq_hz=250,
    start_freq_jitter_hz=20,
    end_freq_hz=800,
    end_freq_jitter_hz=50,
    num_contour_points=3,
    contour_variability_hz=10,
    min_harmonics=20,
    max_harmonics=25,
    harmonic_decay_db=1,
    vibrato_rate_hz=2.5,
    vibrato_depth_hz=1.0,
    sub_harmonic_ratios=[0.5],
    sub_harmonic_amplitude_ratio=0.25,
    add_breathy_noise=True,
    breathy_noise_amount=0.15,
    breathy_noise_lp_cutoff_hz=1800,
    low_cutoff_hz=200,
    high_cutoff_hz=5000,
    envelope_taper_ratio=0.8,
    effects=[reverb_effect],
)

whale_calls_complex = whale_signal_model.generate(
    source=whale_source,
    sensor_delays_s=np.array([0.0]),
    tloss_db=90.0,
    propagation_time_s=0.0,
)
whale_calls_real = np.real(whale_calls_complex[0, :])
component_signals["whale_call"] = whale_calls_real

preview_signal(
    whale_calls_real,
    title="Whale Call Spectrogram",
    n_fft=4096,
    hop_length=1024,
    y_lim=(0, 5500),
)

## Snapping Shrimp Signal

This section models the broadband crackle associated with a snapping shrimp colony.

- The snap is short, impulsive, and strongly broadband.
- This makes it a useful contrast against the tonal or structured signals elsewhere in the notebook.
- The example uses a point-source far-field approximation rather than a diffuse colony model.


In [ ]:
shrimp_source = GroundTruthState(
    [0, 0, 0, 0, -50, 0],
    timestamp=REFERENCE_TIME,
    metadata={
        "amplitude_upa": 10 ** (195 / 20),
        "position_mapping": [0, 2, 4],
    },
)

shrimp_signal_model = PointSourceSnappingShrimpSignal(
    duration_s=SIGNAL_DURATION_S,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    temperature_celsius=25,
    start_time_hours=18.0,
    diurnal_amplitude=0.25,
    diurnal_phase_hours=6,
    delay_duration=0.0006,
    onset_duration=0.0001,
    snap_duration=0.0014,
    onset_level=0.15,
    onset_freq=2500,
    snap_decay=1000,
    low_cutoff_hz=1500,
    high_cutoff_hz=10000,
)

shrimp_signal = shrimp_signal_model.generate(
    source=shrimp_source,
    sensor_delays_s=np.array([0.0]),
    tloss_db=80.0,
    propagation_time_s=10.0,
)
shrimp_signal_real = np.real(shrimp_signal[0, :])
component_signals["snapping_shrimp"] = shrimp_signal_real

preview_signal(
    shrimp_signal_real,
    title="Snapping Shrimp Spectrogram",
    n_fft=2048,
    hop_length=512,
    y_lim=(0, 20000),
)

## Commercial Vessel Tonals

This section generates a simplified low-frequency ship signature.

- The signal is dominated by a handful of narrowband tonal lines.
- These tones represent blade-rate and machinery components.
- In contrast to the biological examples above, this source is continuous and spectrally stable over the window shown here.


In [ ]:
commercial_vessel_state = GroundTruthState(
    [0, 0, 0, 0, -10, 0],
    timestamp=REFERENCE_TIME,
    metadata={
        "frequencies_hz": np.array([50.0, 75.0, 125.0, 82.0]),
        "amplitudes_upa": 10 ** (np.array([175.0, 168.0, 162.0, 160.0]) / 20),
        "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
        "position_mapping": [0, 2, 4],
    },
)

tonal_signal = NarrowbandTonalSignal(
    duration_s=SIGNAL_DURATION_S,
    sampling_rate_hz=SAMPLING_RATE_HZ,
).generate(
    source=commercial_vessel_state,
    sensor_delays_s=np.array([0.0]),
    tloss_db=60.0,
    propagation_time_s=0.0,
)

tonal_signal_real = np.real(tonal_signal[0, :])
component_signals["commercial_vessel"] = tonal_signal_real

preview_signal(
    tonal_signal_real,
    title="Commercial Vessel Tonal Spectrogram",
    n_fft=4096 * 6,
    hop_length=1024,
    y_lim=(0, 200),
    yaxis_format="Hz",
)

## Measured Vessel Noise

This section loads a real recording of a commercial vessel from a WAV file. This recording came from [Sanct Sounds](https://sanctsound.ioos.us/sounds.html#Vessels) and contains the recording of a large vessel.

In [ ]:
from pathlib import Path

wav_name = "SanctSound_CI05_03_largeship_20190925T135956Z.wav"
measured_wav_path = Path("measured_data") / wav_name  # for notebooks

measured_vessel_state = GroundTruthState(
    [0, 0, 0, 0, -10, 0],
    timestamp=REFERENCE_TIME,
    metadata={
        "frequencies_hz": np.array([50.0, 75.0, 125.0, 82.0]),
        "amplitudes_upa": 10 ** (np.array([175.0, 168.0, 162.0, 160.0]) / 20),
        "phases_rad": np.random.uniform(0, 2 * np.pi, 4),
        "position_mapping": [0, 2, 4],
    },
)

measured_signal_model = BroadbandRecordedSignal(
    duration_s=SIGNAL_DURATION_S,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    frame_len=500,
    hop_factor=2,
    wav_path=str(measured_wav_path),
    segment_start_s=0.0,
    segment_duration_s=30.0,
    duration_match_mode="tile",
    level_db_re_1upa=85.0,
)

# BroadbandRecordedSignal is frequency-domain only
_ = measured_signal_model.compute_stft(source=measured_vessel_state)
measured_signal_real = np.real(measured_signal_model.get_source_signal())

component_signals["measured_vessel"] = measured_signal_real

preview_signal(
    measured_signal_real,
    title="Commercial Vessel Tonal Spectrogram",
    n_fft=4096 * 6,
    hop_length=1024,
    y_lim=(0, 4000),
    yaxis_format="Hz",
)

## Ambient White Noise

This section generates a simple ambient baseline.

- `WhiteNoise` is used here as a deliberately simple reference model.
- It does not attempt to reproduce a full ocean ambient spectrum.
- The output is useful as a baseline when contrasting structured and unstructured energy.


In [ ]:
ambient_noise = WhiteNoise(
    amplitude_upa=10 ** (90 / 20),
    duration_s=SIGNAL_DURATION_S,
    sampling_rate_hz=SAMPLING_RATE_HZ,
).generate()

ambient_noise_real = np.real(ambient_noise[0, :])
component_signals["ambient_white_noise"] = ambient_noise_real

preview_signal(
    ambient_noise_real,
    title="Ambient White Noise Spectrogram",
    n_fft=4096,
    hop_length=1024,
    y_lim=(0, 20000),
)

## Composite Soundscape

This final signal combines the individual components into one simple scene.

- The vessel tonals dominate the low end.
- The whale call contributes mid-band contour and harmonic structure.
- The shrimp and white-noise components raise the broadband floor.


In [ ]:
soundscape = (
    component_signals["whale_call"]
    + component_signals["snapping_shrimp"]
    + component_signals["commercial_vessel"]
    + component_signals["measured_vessel"]
    + component_signals["ambient_white_noise"]
)
component_signals["composite_soundscape"] = soundscape

preview_signal(
    soundscape,
    title="Composite Soundscape Spectrogram",
    n_fft=4096,
    hop_length=1024,
    y_lim=(0, 5500),
)